# 100 m Data Foundation

This notebook builds the first reproducible 100 m data foundation for the thesis analysis in Styria.

It deliberately does **not** regenerate or modify the firm input dataset. The existing firm GeoParquet is read as an input and an analytical copy is created with 100 m raster assignments.

The workflow only uses the 100 m resolution. The planned 500 m and 1 km variants are ignored here to keep the workflow simple and avoid additional population-backcasting complexity.

## Inputs and Outputs

Required inputs:

- `OGD/Gemeindegrenzen.zip`: municipality boundaries for Styria.
- `OGD/population_grid_styria.geoparquet`: clipped 2025 100 m population grid for Styria.
- `SDG/companies_styria_syn.geoparquet`: current firm point input for pipeline development.

Optional input for population backcasting:

- `OGD/STMK_POP_2002_2025.csv`

Expected columns for the optional population file:

- `LAU_CODE`
- `POP_2015` through `POP_2025`

Outputs written by this notebook:

- `ANAL/data/raster_100m_styria.geoparquet`
- `ANAL/data/firms_assigned_100m.geoparquet`
- `ANAL/data/population_backcast_100m_quarterly.parquet`, only if the municipal population CSV exists.
- `ANAL/data/raster_quarter_panel_100m.parquet`

## Methodological Notes

The full 100 m raster universe is generated from the Styria municipality boundaries in EPSG:3035. Every grid cell receives a stable `grid_id`, a municipality assignment, and the 2025 population value where available.

Municipality assignment is done in two steps. First, cell centroids are joined to municipalities. For rare unresolved border cases, the municipality with the largest cell intersection is used.

The first raster-quarter panel is built for cells that are relevant at this stage: cells with 2025 population or at least one firm. A fully routable active-cell universe will be defined later after Valhalla routing products exist.

## 1. Imports and Paths

In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box

PROJECT_DIR = Path.cwd().resolve().parents[0]
OGD_DIR = PROJECT_DIR / "OGD"
SDG_DIR = PROJECT_DIR / "SDG"
ANAL_DIR = PROJECT_DIR / "ANAL"
OUTPUT_DIR = ANAL_DIR / "data"

MUNICIPALITIES_PATH = OGD_DIR / "Gemeindegrenzen.zip"
POPULATION_2025_PATH = OGD_DIR / "population_grid_styria.geoparquet"
FIRMS_INPUT_PATH = SDG_DIR / "companies_styria_syn.geoparquet"
MUNICIPAL_POPULATION_PATH = OGD_DIR / "STMK_POP_2002_2025.csv"

RASTER_OUTPUT = OUTPUT_DIR / "raster_100m_styria.geoparquet"
FIRMS_OUTPUT = OUTPUT_DIR / "firms_assigned_100m.geoparquet"
POPULATION_BACKCAST_OUTPUT = OUTPUT_DIR / "population_backcast_100m_quarterly.parquet"
PANEL_OUTPUT = OUTPUT_DIR / "raster_quarter_panel_100m.parquet"

CRS = "EPSG:3035"
CELL_SIZE = 100
START_YEAR = 2015
END_YEAR = 2025
CENSORING_DATE = pd.Timestamp("2025-12-31")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
MUNICIPALITIES_PATH

WindowsPath('D:/CO2_Masterarbeit/CO2_Masterarbeit/OGD/Gemeindegrenzen.zip')

## 2. Helper Functions

In [3]:
def read_zipped_shapefile(zip_path: Path) -> gpd.GeoDataFrame:
    """Read a zipped shapefile reliably on Windows and Linux."""
    archive_path = zip_path.resolve().as_posix()
    zip_uri = f"zip://{archive_path}"
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*FLAECHE_HA parsed incompletely.*")
        return gpd.read_file(zip_uri)


def make_grid_id(easting: pd.Series, northing: pd.Series) -> pd.Series:
    return "AT_CRS3035RES100mN" + northing.astype("int64").astype(str) + "E" + easting.astype("int64").astype(str)


def date_to_quarter(value: pd.Series) -> pd.Series:
    dates = pd.to_datetime(value, errors="coerce")
    return dates.dt.to_period("Q").astype("string")


def check(condition: bool, message: str) -> None:
    status = "OK" if condition else "CHECK"
    print(f"{status}: {message}")

## 3. Load Existing Inputs

The inputs are inspected before creating derived products. The firm input file is treated as read-only.

In [4]:
municipalities = read_zipped_shapefile(MUNICIPALITIES_PATH).to_crs(CRS)
municipalities = municipalities[["GEMNR6", "GEMNR", "GEMNAM", "geometry"]].copy()
municipalities = municipalities.rename(
    columns={
        "GEMNR6": "municipality_id",
        "GEMNR": "municipality_short_id",
        "GEMNAM": "municipality_name",
    }
)
municipalities["municipality_id"] = municipalities["municipality_id"].astype(str)

population_2025 = gpd.read_parquet(POPULATION_2025_PATH).to_crs(CRS)
population_2025["population"] = population_2025["population"].fillna(0).astype(float)

firms_source = gpd.read_parquet(FIRMS_INPUT_PATH).to_crs(CRS)

print(f"Municipalities: {len(municipalities):,}")
print(f"2025 population cells with population: {len(population_2025):,}")
print(f"Firm input rows: {len(firms_source):,}")
print(f"Municipality CRS: {municipalities.crs}")
print(f"Population CRS: {population_2025.crs}")
print(f"Firm CRS: {firms_source.crs}")

Municipalities: 285
2025 population cells with population: 123,403
Firm input rows: 10,000
Municipality CRS: EPSG:3035
Population CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89-extended / LAEA Europe", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}, {"na

## 4. Build the Full 100 m Raster Universe

The raster is created from the total Styria boundary. It includes zero-population cells so later analyses can define active cells consistently.

In [5]:
minx, miny, maxx, maxy = municipalities.total_bounds

start_x = int(np.floor(minx / CELL_SIZE) * CELL_SIZE)
end_x = int(np.ceil(maxx / CELL_SIZE) * CELL_SIZE)
start_y = int(np.floor(miny / CELL_SIZE) * CELL_SIZE)
end_y = int(np.ceil(maxy / CELL_SIZE) * CELL_SIZE)

eastings = np.arange(start_x, end_x, CELL_SIZE)
northings = np.arange(start_y, end_y, CELL_SIZE)
xx, yy = np.meshgrid(eastings, northings)

candidate_cells = pd.DataFrame(
    {
        "easting": xx.ravel(),
        "northing": yy.ravel(),
    }
)
candidate_cells["centroid_x"] = candidate_cells["easting"] + CELL_SIZE / 2
candidate_cells["centroid_y"] = candidate_cells["northing"] + CELL_SIZE / 2

candidate_points = gpd.GeoDataFrame(
    candidate_cells,
    geometry=gpd.points_from_xy(candidate_cells["centroid_x"], candidate_cells["centroid_y"]),
    crs=CRS,
)

inside_styria = gpd.sjoin(
    candidate_points,
    municipalities[["municipality_id", "geometry"]],
    how="inner",
    predicate="within",
).drop(columns=["index_right", "municipality_id"])

inside_styria = inside_styria.drop_duplicates(subset=["easting", "northing"]).reset_index(drop=True)
inside_styria["geometry"] = [
    box(easting, northing, easting + CELL_SIZE, northing + CELL_SIZE)
    for easting, northing in zip(inside_styria["easting"], inside_styria["northing"])
]

raster = gpd.GeoDataFrame(inside_styria, geometry="geometry", crs=CRS)
raster["grid_id"] = make_grid_id(raster["easting"], raster["northing"])
raster["area_m2"] = raster.geometry.area.round(6)
raster["resolution_m"] = CELL_SIZE

print(f"Candidate 100 m cells in bounding box: {len(candidate_cells):,}")
print(f"100 m raster cells with centroids inside Styria municipalities: {len(raster):,}")
display(raster.head())

Candidate 100 m cells in bounding box: 2,671,272
100 m raster cells with centroids inside Styria municipalities: 1,641,287


,easting,northing,centroid_x,centroid_y,geometry,grid_id,area_m2,resolution_m
0,4740200,2626400,4740250.0,2626450.0,"POLYGON ((4740300 2626400, 4740300 2626500, 47...",AT_CRS3035RES100mN2626400E4740200,10000.0,100
1,4739900,2626500,4739950.0,2626550.0,"POLYGON ((4740000 2626500, 4740000 2626600, 47...",AT_CRS3035RES100mN2626500E4739900,10000.0,100
2,4740000,2626500,4740050.0,2626550.0,"POLYGON ((4740100 2626500, 4740100 2626600, 47...",AT_CRS3035RES100mN2626500E4740000,10000.0,100
3,4740100,2626500,4740150.0,2626550.0,"POLYGON ((4740200 2626500, 4740200 2626600, 47...",AT_CRS3035RES100mN2626500E4740100,10000.0,100
4,4740200,2626500,4740250.0,2626550.0,"POLYGON ((4740300 2626500, 4740300 2626600, 47...",AT_CRS3035RES100mN2626500E4740200,10000.0,100


## 5. Assign Municipalities

Centroid assignment is used first because it is simple and reproducible. If a border cell centroid is outside all municipality polygons, the fallback assigns the cell to the municipality with the largest intersecting area.

In [6]:
centroids = raster[["grid_id", "geometry"]].copy()
centroids["geometry"] = centroids.geometry.centroid

centroid_join = gpd.sjoin(
    centroids,
    municipalities[["municipality_id", "municipality_name", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])

raster = raster.merge(
    centroid_join[["grid_id", "municipality_id", "municipality_name"]],
    on="grid_id",
    how="left",
)

missing_municipality = raster["municipality_id"].isna()
print(f"Cells without centroid municipality assignment: {missing_municipality.sum():,}")

if missing_municipality.any():
    unresolved = raster.loc[missing_municipality, ["grid_id", "geometry"]].copy()
    intersections = gpd.overlay(
        unresolved,
        municipalities[["municipality_id", "municipality_name", "geometry"]],
        how="intersection",
    )
    intersections["intersection_area"] = intersections.geometry.area
    fallback = intersections.sort_values("intersection_area", ascending=False).drop_duplicates("grid_id")
    fallback = fallback[["grid_id", "municipality_id", "municipality_name"]]
    raster = raster.drop(columns=["municipality_id", "municipality_name"]).merge(
        pd.concat(
            [
                centroid_join.dropna(subset=["municipality_id"])[["grid_id", "municipality_id", "municipality_name"]],
                fallback,
            ],
            ignore_index=True,
        ),
        on="grid_id",
        how="left",
    )

print(f"Cells still without municipality assignment: {raster['municipality_id'].isna().sum():,}")

Cells without centroid municipality assignment: 0
Cells still without municipality assignment: 0


## 6. Attach 2025 Population

The clipped POPREG grid contains only populated 100 m cells. Missing population values in the full raster are therefore set to zero.

In [7]:
population_table = population_2025[["cell_id", "population"]].rename(
    columns={"cell_id": "grid_id", "population": "population_2025"}
)

raster = raster.merge(population_table, on="grid_id", how="left")
raster["population_2025"] = raster["population_2025"].fillna(0).astype(float)

raster = raster[
    [
        "grid_id",
        "easting",
        "northing",
        "centroid_x",
        "centroid_y",
        "municipality_id",
        "municipality_name",
        "area_m2",
        "resolution_m",
        "population_2025",
        "geometry",
    ]
]

raster.to_parquet(RASTER_OUTPUT, index=False)
print(f"Saved raster universe: {RASTER_OUTPUT}")
print(f"Total 2025 population joined to full raster: {raster['population_2025'].sum():,.0f}")

Saved raster universe: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\raster_100m_styria.geoparquet
Total 2025 population joined to full raster: 1,271,674


## 7. Assign Firms to 100 m Cells

The source firm dataset is not changed. This section creates an analytical copy with raster IDs and quarter variables.

In [8]:
firms = firms_source.copy()
firms["firm_id"] = np.arange(1, len(firms) + 1)
firms["founding_date"] = pd.to_datetime(firms["Mitglied_Gründungsdatum"], errors="coerce")
firms["exit_date"] = pd.to_datetime(firms["Mitglied_Löschdatum"], errors="coerce")
firms["founding_quarter"] = date_to_quarter(firms["founding_date"])
firms["exit_quarter"] = date_to_quarter(firms["exit_date"])
firms["exit_observed"] = firms["exit_date"].notna()

firm_join = gpd.sjoin(
    firms[["firm_id", "geometry"]],
    raster[["grid_id", "municipality_id", "municipality_name", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])

firms = firms.merge(
    firm_join[["firm_id", "grid_id", "municipality_id", "municipality_name"]],
    on="firm_id",
    how="left",
)
firms = firms.rename(columns={"grid_id": "grid_id_100m"})

firms.to_parquet(FIRMS_OUTPUT, index=False)
print(f"Saved firm assignment: {FIRMS_OUTPUT}")
print(f"Firms without 100 m grid assignment: {firms['grid_id_100m'].isna().sum():,}")
display(firms.head())

Saved firm assignment: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\firms_assigned_100m.geoparquet
Firms without 100 m grid assignment: 3


,Sparte_ID,Sparte_Text,Fachgruppe_ID,Fachgruppe_Text,Mitglied_Gründungsdatum,Standort_Angelegt,Standort_Gelöscht,Mitglied_Löschdatum,OSM_Building_ID,OSM_Building_Type,...,geometry,firm_id,founding_date,exit_date,founding_quarter,exit_quarter,exit_observed,grid_id_100m,municipality_id,municipality_name
0,1,Gewerbe und Handwerk,128,FG Persönliche Dienstleister,1963-03-26,1973-07-09,NaT,NaT,391722827,yes,...,POINT (4746259.779 2695046.029),1,1963-03-26,NaT,1963Q1,<NA>,False,AT_CRS3035RES100mN2695000E4746200,61766,Weiz
1,1,Gewerbe und Handwerk,124,LI Friseure,1977-06-25,2002-01-08,NaT,NaT,334682208,yes,...,POINT (4748507.322 2650071.709),2,1977-06-25,NaT,1977Q2,<NA>,False,AT_CRS3035RES100mN2650000E4748500,61008,Gabersdorf
2,6,Tourismus und Freizeitwirtschaft,606,FG Freizeit- und Sportbetriebe,1957-07-30,1991-04-09,NaT,NaT,406245160,yes,...,POINT (4764583.347 2693758.785),3,1957-07-30,NaT,1957Q3,<NA>,False,AT_CRS3035RES100mN2693700E4764500,62266,Feistritztal
3,1,Gewerbe und Handwerk,119,LI Lebensmittelgewerbe,1973-02-17,1984-01-19,2008-10-16,2022-05-17,355497855,yes,...,POINT (4607515.176 2707186.655),4,1973-02-17,2022-05-17,1973Q1,2022Q2,True,AT_CRS3035RES100mN2707100E4607500,61217,Haus
4,1,Gewerbe und Handwerk,126,FG Gewerbliche Dienstleister,1986-06-28,2021-06-22,2021-07-19,NaT,142166061,yes,...,POINT (4748672.642 2683411.335),5,1986-06-28,NaT,1986Q2,<NA>,False,AT_CRS3035RES100mN2683400E4748600,60661,Eggersdorf bei Graz


## 8. Build the Population Backcast if Municipal Counts Exist

The backcast uses the 2025 100 m population distribution as the small-scale baseline and scales it by annual municipal population development.

Formula for cell `i`, municipality `g`, and year `y`:

`population_i_y = population_i_2025 * (population_g_y / population_g_2025)`

Annual values are used for all four quarters of the same year to avoid false quarterly precision.

In [9]:
population_backcast = None

if not MUNICIPAL_POPULATION_PATH.exists():
    print("Population backcast skipped.")
    print(f"Missing optional input: {MUNICIPAL_POPULATION_PATH}")
    print("Expected file: semicolon-separated STMK population CSV with LAU_CODE and POP_2015 through POP_2025")
else:
    municipal_population_wide = pd.read_csv(MUNICIPAL_POPULATION_PATH, sep=";", encoding="cp1252")
    population_columns = [f"POP_{year}" for year in range(START_YEAR, END_YEAR + 1)]
    required_columns = ["LAU_CODE", "LAU_NAME", *population_columns]
    missing_columns = [column for column in required_columns if column not in municipal_population_wide.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in {MUNICIPAL_POPULATION_PATH.name}: {missing_columns}")

    municipal_population = municipal_population_wide[required_columns].copy()
    municipal_population["municipality_id"] = municipal_population["LAU_CODE"].astype(str)
    municipal_population = municipal_population.melt(
        id_vars=["municipality_id", "LAU_NAME"],
        value_vars=population_columns,
        var_name="year",
        value_name="population",
    )
    municipal_population["year"] = municipal_population["year"].str.replace("POP_", "", regex=False).astype(int)
    municipal_population["population"] = pd.to_numeric(municipal_population["population"], errors="coerce")

    expected_years = set(range(START_YEAR, END_YEAR + 1))
    actual_years = set(municipal_population["year"].unique())
    missing_years = sorted(expected_years - actual_years)
    if missing_years:
        raise ValueError(f"Missing population years: {missing_years}")

    pop_2025_municipality = (
        raster.groupby("municipality_id", as_index=False)["population_2025"]
        .sum()
        .rename(columns={"population_2025": "municipality_population_2025"})
    )
    municipal_population = municipal_population.merge(pop_2025_municipality, on="municipality_id", how="left")
    missing_grid_population = municipal_population[municipal_population["municipality_population_2025"].isna()]
    if not missing_grid_population.empty:
        missing_ids = sorted(missing_grid_population["municipality_id"].unique())
        raise ValueError(f"Municipalities missing from raster assignment: {missing_ids[:10]}")
    municipal_population["scaling_factor"] = municipal_population["population"] / municipal_population["municipality_population_2025"]

    base_cells = raster[["grid_id", "municipality_id", "population_2025"]].copy()
    annual_backcast = base_cells.merge(municipal_population, on="municipality_id", how="left")
    annual_backcast["population_backcast"] = annual_backcast["population_2025"] * annual_backcast["scaling_factor"]
    annual_backcast = annual_backcast.rename(columns={"population": "municipality_population_year"})

    quarters = pd.DataFrame({"quarter": [1, 2, 3, 4]})
    population_backcast = annual_backcast.merge(quarters, how="cross")
    population_backcast["backcast_method"] = "annual_municipal_scaling_from_2025_100m_grid"
    population_backcast = population_backcast[
        [
            "grid_id",
            "year",
            "quarter",
            "municipality_id",
            "population_backcast",
            "population_2025",
            "municipality_population_year",
            "municipality_population_2025",
            "scaling_factor",
            "backcast_method",
        ]
    ]
    population_backcast.to_parquet(POPULATION_BACKCAST_OUTPUT, index=False)
    print(f"Saved population backcast: {POPULATION_BACKCAST_OUTPUT}")

Saved population backcast: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\population_backcast_100m_quarterly.parquet


In [10]:
population_backcast

,grid_id,year,quarter,municipality_id,population_backcast,population_2025,municipality_population_year,municipality_population_2025,scaling_factor,backcast_method
0,AT_CRS3035RES100mN2626400E4740200,2015,1,61054,0.0,0.0,3778,3492.0,1.081901,annual_municipal_scaling_from_2025_100m_grid
1,AT_CRS3035RES100mN2626400E4740200,2015,2,61054,0.0,0.0,3778,3492.0,1.081901,annual_municipal_scaling_from_2025_100m_grid
2,AT_CRS3035RES100mN2626400E4740200,2015,3,61054,0.0,0.0,3778,3492.0,1.081901,annual_municipal_scaling_from_2025_100m_grid
3,AT_CRS3035RES100mN2626400E4740200,2015,4,61054,0.0,0.0,3778,3492.0,1.081901,annual_municipal_scaling_from_2025_100m_grid
4,AT_CRS3035RES100mN2626400E4740200,2016,1,61054,0.0,0.0,3794,3492.0,1.086483,annual_municipal_scaling_from_2025_100m_grid
...,...,...,...,...,...,...,...,...,...,...
72216623,AT_CRS3035RES100mN2760400E4720600,2024,4,62142,0.0,0.0,3623,3538.0,1.024025,annual_municipal_scaling_from_2025_100m_grid
72216624,AT_CRS3035RES100mN2760400E4720600,2025,1,62142,0.0,0.0,3547,3538.0,1.002544,annual_municipal_scaling_from_2025_100m_grid
72216625,AT_CRS3035RES100mN2760400E4720600,2025,2,62142,0.0,0.0,3547,3538.0,1.002544,annual_municipal_scaling_from_2025_100m_grid
72216626,AT_CRS3035RES100mN2760400E4720600,2025,3,62142,0.0,0.0,3547,3538.0,1.002544,annual_municipal_scaling_from_2025_100m_grid


## 9. Build the 100 m Raster-Quarter Panel

The first panel uses a preliminary analytical cell set: cells with positive 2025 population or at least one firm. This avoids a very large all-cell panel before the routable active-cell definition exists.

In [11]:
firm_cell_ids = firms["grid_id_100m"].dropna().unique()
panel_cells = raster[(raster["population_2025"] > 0) | (raster["grid_id"].isin(firm_cell_ids))].copy()
panel_cells["preliminary_panel_cell"] = True

quarters = pd.period_range(f"{START_YEAR}Q1", f"{END_YEAR}Q4", freq="Q")
quarter_table = pd.DataFrame(
    {
        "period": quarters.astype(str),
        "year": quarters.year,
        "quarter": quarters.quarter,
    }
)

panel = panel_cells[["grid_id", "municipality_id", "municipality_name", "population_2025", "preliminary_panel_cell"]].merge(
    quarter_table,
    how="cross",
)

analysis_start = pd.Timestamp(f"{START_YEAR}-01-01")

births = (
    firms.dropna(subset=["grid_id_100m", "founding_quarter"])
    .loc[lambda df: df["founding_date"].between(analysis_start, CENSORING_DATE, inclusive="both")]
    .groupby(["grid_id_100m", "founding_quarter"], as_index=False)
    .size()
    .rename(columns={"grid_id_100m": "grid_id", "founding_quarter": "period", "size": "births"})
)

exits = (
    firms.dropna(subset=["grid_id_100m", "exit_quarter"])
    .loc[lambda df: df["exit_date"].between(analysis_start, CENSORING_DATE, inclusive="both")]
    .groupby(["grid_id_100m", "exit_quarter"], as_index=False)
    .size()
    .rename(columns={"grid_id_100m": "grid_id", "exit_quarter": "period", "size": "exits"})
)

panel = panel.merge(births, on=["grid_id", "period"], how="left")
panel = panel.merge(exits, on=["grid_id", "period"], how="left")
panel["births"] = panel["births"].fillna(0).astype(int)
panel["exits"] = panel["exits"].fillna(0).astype(int)

firm_dates = firms.dropna(subset=["grid_id_100m", "founding_date"])[["firm_id", "grid_id_100m", "founding_date", "exit_date"]].copy()
firm_dates["exit_date_filled"] = firm_dates["exit_date"].fillna(pd.Timestamp.max)

active_records = []
for period in quarters:
    quarter_end = period.end_time.normalize()
    previous_quarter_end = (period - 1).end_time.normalize()
    active_now = firm_dates[(firm_dates["founding_date"] <= quarter_end) & (firm_dates["exit_date_filled"] > quarter_end)]
    active_previous = firm_dates[(firm_dates["founding_date"] <= previous_quarter_end) & (firm_dates["exit_date_filled"] > previous_quarter_end)]

    active_now_counts = active_now.groupby("grid_id_100m").size().rename("active_firms_t")
    active_previous_counts = active_previous.groupby("grid_id_100m").size().rename("active_firms_tminus1")
    active_counts = pd.concat([active_now_counts, active_previous_counts], axis=1).fillna(0).astype(int).reset_index()
    active_counts = active_counts.rename(columns={"grid_id_100m": "grid_id"})
    active_counts["period"] = str(period)
    active_records.append(active_counts)

active_panel = pd.concat(active_records, ignore_index=True)
panel = panel.merge(active_panel, on=["grid_id", "period"], how="left")
panel["active_firms_t"] = panel["active_firms_t"].fillna(0).astype(int)
panel["active_firms_tminus1"] = panel["active_firms_tminus1"].fillna(0).astype(int)

if population_backcast is None and POPULATION_BACKCAST_OUTPUT.exists():
    population_backcast = pd.read_parquet(POPULATION_BACKCAST_OUTPUT)

if population_backcast is not None:
    panel = panel.merge(
        population_backcast[["grid_id", "year", "quarter", "population_backcast"]],
        on=["grid_id", "year", "quarter"],
        how="left",
    )
else:
    panel["population_backcast"] = pd.NA

panel.to_parquet(PANEL_OUTPUT, index=False)
print(f"Saved raster-quarter panel: {PANEL_OUTPUT}")
print(f"Panel cells: {panel_cells['grid_id'].nunique():,}")
print(f"Panel rows: {len(panel):,}")

Saved raster-quarter panel: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\raster_quarter_panel_100m.parquet
Panel cells: 125,473
Panel rows: 5,520,812


## 10. Validation Summary

In [12]:
print("Raster checks")
check(raster.crs.to_epsg() == 3035, "raster CRS is EPSG:3035")
check(raster["grid_id"].is_unique, "grid_id is unique")
check(raster["municipality_id"].notna().all(), "all raster cells have municipality_id")
check((raster["resolution_m"] == 100).all(), "all cells are marked as 100 m")
check(np.isclose(raster["population_2025"].sum(), population_2025["population"].sum()), "2025 population sum matches clipped population grid")

print("\nFirm checks")
check(len(firms_source) > 0, "source firm input has rows")
check(len(firms) == len(firms_source), "analytical firm output has the same number of rows as the input")
check(firms["grid_id_100m"].notna().all(), "all firms receive grid_id_100m")
invalid_exit_dates = firms["exit_date"].notna() & (firms["exit_date"] < firms["founding_date"])
check(not invalid_exit_dates.any(), "no exit date before founding date")

print("\nPopulation backcast checks")
if population_backcast is None:
    print("CHECK: population backcast not created because the population input file is missing")
else:
    validation = (
        population_backcast.drop_duplicates(["grid_id", "year"])
        .groupby(["municipality_id", "year"], as_index=False)
        .agg(
            raster_sum=("population_backcast", "sum"),
            official_population=("municipality_population_year", "first"),
            scaling_factor=("scaling_factor", "first"),
        )
    )
    validation["absolute_deviation"] = validation["raster_sum"] - validation["official_population"]
    validation["relative_deviation"] = validation["absolute_deviation"] / validation["official_population"]
    display(validation.sort_values("absolute_deviation", key=lambda s: s.abs(), ascending=False).head(10))
    suspicious = validation[(validation["scaling_factor"] < 0.75) | (validation["scaling_factor"] > 1.25)]
    print(f"Suspicious scaling factors outside 0.75-1.25: {len(suspicious):,}")

print("\nPanel checks")
expected_rows = panel["grid_id"].nunique() * len(quarters)
check(len(panel) == expected_rows, "panel has one row per preliminary panel cell and quarter")
check(panel["period"].min() == f"{START_YEAR}Q1" and panel["period"].max() == f"{END_YEAR}Q4", "panel covers 2015Q1-2025Q4")
expected_births = firms[firms["founding_date"].between(pd.Timestamp(f"{START_YEAR}-01-01"), CENSORING_DATE, inclusive="both")].shape[0]
expected_exits = firms[firms["exit_date"].between(pd.Timestamp(f"{START_YEAR}-01-01"), CENSORING_DATE, inclusive="both")].shape[0]
check(panel["births"].sum() == expected_births, "panel births match firm events in period")
check(panel["exits"].sum() == expected_exits, "panel exits match firm events in period")
check((panel["active_firms_tminus1"] >= 0).all(), "active_firms_tminus1 is never negative")

Raster checks
OK: raster CRS is EPSG:3035
OK: grid_id is unique
OK: all raster cells have municipality_id
OK: all cells are marked as 100 m
CHECK: 2025 population sum matches clipped population grid

Firm checks
OK: source firm input has rows
OK: analytical firm output has the same number of rows as the input
CHECK: all firms receive grid_id_100m
OK: no exit date before founding date

Population backcast checks


,municipality_id,year,raster_sum,official_population,scaling_factor,absolute_deviation,relative_deviation
2379,62140,2018,22798.0,22798,1.044726,3.637979e-12,1.595745e-16
937,61108,2017,24915.0,24915,1.015654,-3.637979e-12,-1.460156e-16
2380,62140,2019,22753.0,22753,1.042663,3.637979e-12,1.598901e-16
2386,62140,2025,21907.0,21907,1.003895,3.637979e-12,1.660647e-16
2376,62140,2015,23188.0,23188,1.062597,-3.637979e-12,-1.568906e-16
504,60664,2024,12879.0,12879,1.006015,-1.818989e-12,-1.412369e-16
499,60664,2019,12931.0,12931,1.010077,1.818989e-12,1.406689e-16
2188,62041,2025,12870.0,12870,1.002024,1.818989e-12,1.413356e-16
2176,62040,2024,9691.0,9691,1.013491,1.818989e-12,1.876988e-16
1672,61631,2015,10093.0,10093,1.059077,-1.818989e-12,-1.802229e-16


Suspicious scaling factors outside 0.75-1.25: 6

Panel checks
OK: panel has one row per preliminary panel cell and quarter
OK: panel covers 2015Q1-2025Q4
OK: panel births match firm events in period
OK: panel exits match firm events in period
OK: active_firms_tminus1 is never negative
